In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, DateType
)
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_date, lit

In [0]:
# ── Schema ──
employee_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),    True),
])

# ── Seed data ──
employee_data = [
    (1, "Aarav Mehta",    "Engineering", 95000.0,  "2022-01-15"),
    (2, "Priya Sharma",   "Marketing",  78000.0,  "2021-06-20"),
    (3, "Rohan Patel",    "Engineering", 102000.0, "2020-03-10"),
    (4, "Sneha Iyer",     "Finance",    88000.0,  "2023-02-01"),
    (5, "Vikram Desai",   "Marketing",  72000.0,  "2022-09-12"),
]

employee_df = spark.createDataFrame(employee_data, employee_schema)

# ── Write as a managed Delta table ──
employee_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.employees_b")

print("✓ employees table created")
spark.table("default.employees_b").display()

In [0]:
# ── Updates / new-hire data ──
update_data = [
    (2, "Priya Sharma",   "Sales",      82000.0,  "2021-06-20"),
    (4, "Sneha Iyer",     "Finance",    95000.0,  "2023-02-01"),
    (6, "Ananya Reddy",   "Engineering", 97000.0, "2024-01-08"),
    (7, "Karthik Nair",   "Finance",    84000.0,  "2024-03-15"),
]

updates_df = spark.createDataFrame(update_data, employee_schema)

updates_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.employee_updates")

print("✓ employee_updates table created")
spark.table("default.employee_updates").display()

Upsert Operation 

In [0]:
target = DeltaTable.forName(spark, "default.employees_b")

source = spark.table("default.employee_updates")

target.alias("t") \
    .merge(
        source.alias("s"),
        "t.emp_id = s.emp_id"  # join condition 
    ) \
    .whenMatchedUpdate(
        set = {
            "name" : col("s.name"),
            "department" : col("s.department"),
            "salary" : col("s.salary"),
            "hire_date" : col("s.hire_date")
        }
    ) \
    .whenNotMatchedInsert(
        values= {
            "emp_id":   col("s.emp_id"),
            "name":     col("s.name"),
            "department" : col("s.department"),
            "salary":   col("s.salary"),
            "hire_date" : col("s.hire_date"),
        }
    ) \
    .execute()

print("✓ MERGE (upsert) complete")
spark.table("default.employees_b").orderBy("emp_id").show()

In [0]:
%sql
MERGE INTO default.employees AS t 
USING default.employee_updates AS s
ON t.emp_id = s.emp_id

WHEN MATCHED THEN 
    UPDATE SET
      t.name  = s.name,
      t.department = s.department,
      t.salary = s.salary,
      t.hire_date = s.hire_date

WHEN NOT MATCHED THEN 
    INSERT (emp_id, name, department, salary, hire_date)
    VALUES (s.emp_id, s.name, s.department, s.salary, s.hire_date);